#**Pruebas de Hipótesis Unimuestrales**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

ruta = '/content/drive/MyDrive/enemdu_consumidor_2026_01 .csv'
df = pd.read_csv(ruta, sep=';')

print(df.head())

   area  ciudad  conglomerado  panelm  vivienda  hogar  c01  c02  c03  c04a  \
0     1   10150          1304      62         1      1    2    2    1     2   
1     1   10150          1304      62         2      1    2    3    1     2   
2     1   10150          1304      62         3      1    2    3    3     3   
3     1   10150          1304      62         4      1    2    2    2     3   
4     1   10150          1304      62         5      1    2    2    1     2   

   ...  c18  c19  c20a  c21a  estrato              fexp          upm  \
0  ...    2    3     2     2     2713  102,542333745242  10150001304   
1  ...    1    2     2     2     2713  102,542333745242  10150001304   
2  ...    3    3     2     2     2713  102,542333745242  10150001304   
3  ...    3    2     2     3     2713  102,542333745242  10150001304   
4  ...    1    2     2     2     2713  102,542333745242  10150001304   

        id_vivienda           id_hogar  periodo  
0  1015000130406201  10150001304062011   2

In [3]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Convertir la columna 'fexp' a numérica, manejando la coma como separador decimal
df['fexp'] = df['fexp'].str.replace(',', '.', regex=False).astype(float)

# Mostrar estadísticas descriptivas de la columna 'fexp' limpia
print("Estadísticas descriptivas para la columna 'fexp':")
print(df['fexp'].describe())

Estadísticas descriptivas para la columna 'fexp':
count     8791.000000
mean       599.367316
std       1238.483664
min          3.782664
25%         92.414099
50%        244.289965
75%        563.976392
max      18136.571643
Name: fexp, dtype: float64


In [4]:
# Seleccionar 'fexp' como la variable cuantitativa estratégica
variable = df['fexp']

# Determinar el tamaño de la muestra (n)
n = len(variable)

# Dado que n es grande (8791), usaremos la distribución t para el cálculo del intervalo de confianza, ya que es más robusta.
# Para n grande, la distribución t se aproxima a la distribución Z.

# Calcular la media y el error estándar
mean_val = np.mean(variable)
std_err = stats.sem(variable)

# Calcular el intervalo de confianza del 95%
confidence_level = 0.95
dof = n - 1  # grados de libertad

# Obtener el valor t-crítico
t_critical = stats.t.ppf((1 + confidence_level) / 2, dof)

# Calcular el margen de error
margin_of_error = t_critical * std_err

# Calcular el intervalo de confianza
conf_interval = (mean_val - margin_of_error, mean_val + margin_of_error)

print(f"Variable Cuantitativa Estratégica: fexp")
print(f"Tamaño de la Muestra (n): {n}")
print(f"Media: {mean_val:.2f}")
print(f"Error Estándar: {std_err:.2f}")
print(f"Intervalo de Confianza del 95%: ({conf_interval[0]:.2f}, {conf_interval[1]:.2f})")

Variable Cuantitativa Estratégica: fexp
Tamaño de la Muestra (n): 8791
Media: 599.37
Error Estándar: 13.21
Intervalo de Confianza del 95%: (573.47, 625.26)


### 1. Prueba de Hipótesis Unimuestral
Definimos la hipótesis nula ($H_0$) y alterna ($H_1$) para el parámetro crítico de la media de consumo:

$$H_0: \mu = 600$$
$$H_1: \mu \neq 600$$

Donde 600 es nuestro valor de referencia para el análisis en la región de Loja.

In [5]:
from scipy.stats import ttest_1samp

# Ejecutamos el test T de una muestra
# 'df['fexp']' es tu columna ya limpia
t_stat, p_value = ttest_1samp(df['fexp'], 600)

print(f"Resultado del Test T:")
print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor-p: {p_value:.4f}")

# Justificación estadística basada en el Valor-p
if p_value < 0.05:
    print("\nDecisión: Como el Valor-p < 0.05, se rechaza la hipótesis nula.")
else:
    print("\nDecisión: Como el Valor-p >= 0.05, no hay evidencia para rechazar la hipótesis nula.")

Resultado del Test T:
Estadístico T: -0.0479
Valor-p: 0.9618

Decisión: Como el Valor-p >= 0.05, no hay evidencia para rechazar la hipótesis nula.


#**2. Comparación de Grupos**

In [6]:
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# --- CONFIGURACIÓN ---
columna_grupos = 'area'  # Puedes cambiar a 'ciudad' o 'parro' si prefieres
variable_objetivo = 'fexp'

# --- 1. ANOVA ---
# Creamos una lista con los datos de cada grupo
grupos = [df[df[columna_grupos] == g][variable_objetivo] for g in df[columna_grupos].unique()]

# Ejecutamos ANOVA de 1 factor
f_stat, p_value_anova = stats.f_oneway(*grupos)

print(f"ANOVA - Valor-p: {p_value_anova:.4f}")

# --- 2. Tukey Post-Hoc ---
# Solo ejecutamos si el ANOVA es significativo (p < 0.05)
if p_value_anova < 0.05:
    print("\nComo el Valor-p < 0.05, existen diferencias significativas. Ejecutando Tukey...")
    tukey = pairwise_tukeyhsd(endog=df[variable_objetivo],
                              groups=df[columna_grupos],
                              alpha=0.05)
    print(tukey)
else:
    print("\nEl Valor-p >= 0.05. No hay diferencias estadísticamente significativas entre los grupos.")

ANOVA - Valor-p: 0.0006

Como el Valor-p < 0.05, existen diferencias significativas. Ejecutando Tukey...
 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper   reject
-----------------------------------------------------
     1      2 103.8459 0.0006 44.7028 162.9891   True
-----------------------------------------------------


Con base en los resultados obtenidos, se concluye que existen diferencias estadísticamente significativas en la variable fexp entre las áreas analizadas. El valor-p del ANOVA (0.0006) es inferior al nivel de significancia de 0.05, lo que permite rechazar la hipótesis nula y confirma que el consumo varía dependiendo del grupo. Adicionalmente, la prueba de Tukey validó una diferencia significativa entre el área 1 y el área 2, con una diferencia de medias de aproximadamente 103.85 unidades, evidenciando una disparidad real en el comportamiento de los datos entre estas categorías.